In [19]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
import os

In [20]:
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key= os.getenv("GROQ_API_KEY"),
    temperature=0,
)

In [21]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [22]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [23]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [24]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()


In [25]:
intial_state = {'title': 'Rise of AI in USA'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of AI in USA', 'outline': '**Blog Title:** *The Rise of AI in the United States: Drivers, Impact, and the Road Ahead*\n\n---\n\n## 1. Introduction  \n- **Hook:** A striking statistic or anecdote (e.g., “In 2024, AI‑driven startups raised over $30\u202fB in U.S. venture capital, a 3‑fold increase from 2019.”)  \n- **Why it matters:** Briefly explain AI’s transformative potential for the economy, national security, and everyday life.  \n- **Scope & Structure:** Outline what the post will cover – historical context, key catalysts, sectoral impacts, policy landscape, challenges, and future outlook.\n\n---\n\n## 2. Historical Context: From Early Research to Mainstream Adoption  \n### 2.1. The Foundations (1950s‑1990s)  \n- Dartmouth Workshop & the birth of AI  \n- Early government programs (DARPA, NSF) and academic hubs (MIT, Stanford)  \n- The “AI winter” periods and lessons learned  \n\n### 2.2. The Turn of the Century (2000‑2015)  \n- Growth of machine learning & big data

In [26]:
print(final_state['outline'])

**Blog Title:** *The Rise of AI in the United States: Drivers, Impact, and the Road Ahead*

---

## 1. Introduction  
- **Hook:** A striking statistic or anecdote (e.g., “In 2024, AI‑driven startups raised over $30 B in U.S. venture capital, a 3‑fold increase from 2019.”)  
- **Why it matters:** Briefly explain AI’s transformative potential for the economy, national security, and everyday life.  
- **Scope & Structure:** Outline what the post will cover – historical context, key catalysts, sectoral impacts, policy landscape, challenges, and future outlook.

---

## 2. Historical Context: From Early Research to Mainstream Adoption  
### 2.1. The Foundations (1950s‑1990s)  
- Dartmouth Workshop & the birth of AI  
- Early government programs (DARPA, NSF) and academic hubs (MIT, Stanford)  
- The “AI winter” periods and lessons learned  

### 2.2. The Turn of the Century (2000‑2015)  
- Growth of machine learning & big data  
- Emergence of cloud computing platforms (AWS, Azure, Google Cl

In [27]:
print(final_state['content'])

# **The Rise of AI in the United States: Drivers, Impact, and the Road Ahead**

*By [Your Name] – [Date]*  

---  

## 1. Introduction  

> **“In 2024, AI‑driven startups raised **$30 billion** in U.S. venture capital—**three times** the amount raised in 2019.”**  
> — *PitchBook AI Funding Report, 2024*  

Artificial Intelligence is no longer a futuristic buzzword; it is reshaping the U.S. economy, national security, and the way we live day‑to‑day. From the voice assistant that orders your groceries to autonomous drones that patrol the skies, AI is the invisible engine powering the next wave of productivity and innovation.  

In this post we’ll trace **how the United States went from a modest research community to a global AI powerhouse**, unpack the **key catalysts** behind the boom, explore **sector‑specific impacts**, dissect the **policy landscape**, and look ahead to the **opportunities and challenges** that will define the next decade.

---

## 2. Historical Context: From Early 